In [ ]:
import pandas as pd

df = pd.read_csv("quality_data.csv")

# Create input text
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

df["input_text"] = df["input_text"].str.lower()

# Create label
df["label"] = df["category"] + "_" + df["subcategory"]

df = df[["input_text", "label"]]
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(
#     df["input_text"],
#     df["label"],
#     test_size=0.2,
#     random_state=42,
#     stratify=df["label"]  # important for balance
# )
X_train = df["input_text"]
y_train = df["label"]

X_test = df["input_text"]
y_test = df["label"]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

tfidf_model.fit(X_train, y_train)

y_pred_tfidf = tfidf_model.predict(X_test)

In [ ]:
print(y_test.value_counts())

In [ ]:
import numpy as np

probs = tfidf_model.predict_proba(X_test)
confidence_scores = np.max(probs, axis=1)

In [ ]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_base")

labels = list(set(y_train))

In [ ]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        return entities[0]["label"], entities[0]["score"]
    return "unknown", 0.0


y_pred_gliner = []
gliner_conf = []

for text in X_test:
    label, score = gliner_predict(text)
    y_pred_gliner.append(label)
    gliner_conf.append(score)

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_pred_tfidf, output_dict=True)

report_df = pd.DataFrame(report).transpose()

# Add F2 score manually
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print(report_df)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)
print("\nAccuracy:", round(accuracy, 3))

In [ ]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())

In [ ]:
result_df = pd.DataFrame({
    "input_text": X_test,
    "actual_label": y_test,
    "predicted_label": y_pred_tfidf,
    "confidence": confidence_scores
}).reset_index(drop=True)

In [ ]:
# Actual split
result_df[["category", "subcategory"]] = result_df["actual_label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [ ]:
result_df["confidence_percent"] = (result_df["confidence"] * 100).round(2).astype(str) + "%"

def confidence_level(c):
    if c >= 0.7:
        return "High"
    elif c >= 0.4:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [ ]:
result_df["correct"] = result_df["actual_label"] == result_df["predicted_label"]

In [ ]:
def extract_fields(text):
    parts = text.split("|")
    narration = parts[0].replace("narration:", "").strip()
    mode = parts[1].replace("mode:", "").strip()
    txn_type = parts[2].replace("type:", "").strip()
    return pd.Series([narration, mode, txn_type])

result_df[["narration", "mode", "type"]] = result_df["input_text"].apply(extract_fields)

In [ ]:
final_df = result_df[
    [
        "narration",
        "mode",
        "type",
        "category",
        "subcategory",
        "predicted_category",
        "predicted_subcategory",
        "confidence",
        "confidence_percent",
        "confidence_level",
        "correct"
    ]
]

final_df.head(20)

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred_tfidf, output_dict=True)
report_df = pd.DataFrame(report).transpose()

In [ ]:
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print("\nClassification Report:")
print(report_df)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])

In [ ]:
# Remove summary rows
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])